In [7]:
using CSV, DataFrames, GLM, StatsPlots, Gurobi, JuMP
using LinearAlgebra, Random, DataFrames, CSV, Plots
using StatsBase, Statistics, Distributions
using JuMP, Gurobi
gurobi_env = Gurobi.Env()


# 1. Read CSV
df = CSV.read("../clean_data/train_data_features_24hr.csv", DataFrame)



# 2. Define target (y) and feature set (X)
# HB_NORTH is your response for holistic regression
y = df.HB_NORTH               # Vector{Float64}

# Drop the target and the timestamp from the features
feature_cols = Not([:HB_NORTH, :interval_start_local])
X_df = df[:, feature_cols]    # DataFrame of predictors

# 3. Handle missing values if there are any (simple example: fill with 0.0)
# You can replace 0.0 with mean/median/etc. if you prefer.
X_df = coalesce.(X_df, 0.0)

# 4. Ensure all features are Float64
X = Matrix{Float64}(X_df)     # n × p matrix of features
y_vec = Vector{Float64}(y)    # n-vector of targets

# 5. (Optional but common) Standardize features for regression
X_std = copy(X)
for j in 1:size(X_std, 2)
    μ = mean(X_std[:, j])
    σ = std(X_std[:, j])
    if σ > 0
        X_std[:, j] .= (X_std[:, j] .- μ) ./ σ
    end
end


# X_test y_test
# 1. Read CSV
df_t = CSV.read("../clean_data/test_data_features_24hr.csv", DataFrame)

timestamps = df_t.interval_start_local


# 2. Define target (y) and feature set (X)
# HB_NORTH is your response for holistic regression
y_t = df_t.HB_NORTH               # Vector{Float64}

# Drop the target and the timestamp from the features
feature_cols = Not([:HB_NORTH, :interval_start_local])
X_df_t = df_t[:, feature_cols]    # DataFrame of predictors

# 3. Handle missing values if there are any (simple example: fill with 0.0)
# You can replace 0.0 with mean/median/etc. if you prefer.
X_df_t = coalesce.(X_df_t, 0.0)

# 4. Ensure all features are Float64
X_t = Matrix{Float64}(X_df_t)     # n × p matrix of features
y_vec_t = Vector{Float64}(y_t)    # n-vector of targets

# 5. (Optional but common) Standardize features for regression
X_std_t = copy(X_t)
for j in 1:size(X_std_t, 2)
    μ = mean(X_std_t[:, j])
    σ = std(X_std_t[:, j])
    if σ > 0
        X_std_t[:, j] .= (X_std_t[:, j] .- μ) ./ σ
    end
end

Set parameter Username
Set parameter LicenseID to value 2749630
Academic license - for non-commercial use only - expires 2026-12-03


In [8]:
using JuMP
using GLPK  # or your favorite solver

"""
    quantile_regression(X, y; tau=0.5)

Solve quantile regression for given tau (median when tau=0.5).
X: Matrix (n, p)
y: Vector length n
"""
function quantile_regression(X, y; tau=0.5)
    n, p = size(X)
    X_design = hcat(ones(n), X)  # (n, p+1)
    p_full = p + 1

    model = Model(GLPK.Optimizer)

    @variable(model, β[1:p_full])
    @variable(model, r_pos[1:n] >= 0)  # max(residual, 0)
    @variable(model, r_neg[1:n] >= 0)  # max(-residual, 0)

    # residual: y - xᵢ'β = r_pos[i] - r_neg[i]
    @constraint(model, [i in 1:n],
        y[i] - sum(X_design[i, j] * β[j] for j in 1:p_full) == r_pos[i] - r_neg[i]
    )

    @objective(model, Min,
        tau * sum(r_pos[i] for i in 1:n) +
        (1 - tau) * sum(r_neg[i] for i in 1:n)
    )

    optimize!(model)

    return value.(β), objective_value(model)
end

# Example:
# β_opt, obj = quantile_regression(X, y; tau=0.5)  # median regression
# println("Intercept: ", β_opt[1])
# println("Slopes: ", β_opt[2:end])

quantile_regression

In [9]:
n_test, p = size(X_std_t)

# Match what we did in training
X_test_design = hcat(ones(n_test), X_std_t)  # (n_test, p+1)


B_qr, obj_qr = quantile_regression(X_std, y, tau=0.5)
test_prices = X_test_design * B_qr;
result = DataFrame(
    timestamp      = timestamps,
    price_q50  = test_prices,   # if this is the median τ = 0.5 model
)
CSV.write("../price_data/lqad_q50.csv", result)

"../price_data/lqad_q50.csv"

In [10]:

B_qr, obj_qr = quantile_regression(X_std, y, tau=0.95)
test_prices = X_test_design * B_qr;
result = DataFrame(
    timestamp      = timestamps,
    price_q50  = test_prices,   # if this is the median τ = 0.5 model
)
CSV.write("../price_data/lqad_q95.csv", result)

"../price_data/lqad_q95.csv"